# KONTOGRAPH — 02: distil the amortised explainer

The vertical spike (ADR-0007).

Perturbation explainers cost **seconds** per instance. The SLA is 200 ms total. That is
two orders of magnitude, not a tuning problem — so the cost is moved to *training* time:
an expensive teacher produces edge masks offline, and a small head learns to predict them
in **one forward pass**.

The risk is that the student learns to produce *plausible* masks rather than
*load-bearing* ones. So this notebook measures the gap rather than assuming distillation
preserved faithfulness, and the gap is published either way.

Run `01_train_tgn` first — this notebook needs its checkpoint.

## Environment

In [ ]:
# PyG needs nothing but PyTorch since 2.3 -- torch-scatter, torch-sparse and
# pyg-lib are optional. So we install torch_geometric plainly and let it use
# whatever torch Kaggle ships, rather than pinning a CUDA wheel index.
#
# Pinning wheels against a CUDA version Kaggle has since moved past is the
# classic way these notebooks rot. Detecting instead of pinning is why this one
# should keep working as the base image changes.
import subprocess, sys, torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

# `cuda.is_available()` is NOT sufficient. Kaggle's P100 is compute capability
# sm_60, and current Kaggle PyTorch builds ship kernels for sm_70 and above --
# so CUDA reports available, every allocation succeeds, and the run dies at the
# first real kernel launch minutes later with an unhelpful error. Checking the
# capability against what this build actually compiled for turns that into an
# immediate, actionable failure.
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    supported = torch.cuda.get_arch_list()
    print(f"device: {name} (sm_{major}{minor})")
    print("this torch build supports:", " ".join(supported))
    if f"sm_{major}{minor}" not in supported:
        raise RuntimeError(
            f"{name} is sm_{major}{minor}, which this PyTorch build does not support "
            f"({' '.join(supported)}).\n"
            "Switch the accelerator: Notebook > Settings > Accelerator > GPU T4 x2 "
            "(T4 is sm_75). P100 is sm_60 and is too old for current Kaggle images.\n"
            "Failing now rather than at the first kernel launch several minutes in."
        )
else:
    raise RuntimeError(
        "No GPU. Notebook > Settings > Accelerator > GPU T4 x2. "
        "GPU and internet both require a phone-verified account."
    )

try:
    import torch_geometric
    print("torch_geometric", torch_geometric.__version__, "(already present)")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
    import torch_geometric
    print("torch_geometric", torch_geometric.__version__, "(installed)")

## Load data, split, graph and model

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, json

# The dataset pushed by `kg kaggle push-data`. Located by CONTENT rather than by
# name: Kaggle does not guarantee where an attached dataset lands. This notebook
# previously globbed "/kaggle/input/*kontograph*" and fell back to
# "/kaggle/input" when that matched nothing -- and it did match nothing, because
# the dataset mounted under a nested "datasets/" directory. The fallback turned
# "the dataset is not attached" into a FileNotFoundError several frames deep in
# pandas, which is a much harder thing to read.
#
# So: search for the file that must exist, and if it does not, say so.
def find_dataset_root(marker="features.parquet"):
    base = Path("/kaggle/input")
    if not base.exists():
        raise RuntimeError("/kaggle/input does not exist -- is this running on Kaggle?")
    for candidate in sorted(base.rglob(marker)):
        return candidate.parent
    listing = [str(p.relative_to(base)) for p in sorted(base.rglob("*")) if p.is_file()][:40]
    raise FileNotFoundError(
        f"no {marker} anywhere under /kaggle/input.\n"
        f"The dataset is not attached, or is still processing.\n"
        f"Attach it via Notebook > Add Input, or re-run `kg kaggle push-kernel`.\n"
        f"What IS mounted: {listing or '(nothing)'}"
    )


ROOT = find_dataset_root()
print("input:", ROOT, "->", sorted(p.name for p in ROOT.iterdir()))

features = pd.read_parquet(ROOT / "features.parquet")
labels = pd.read_parquet(ROOT / "labels.parquet")
manifest = json.loads((ROOT / "manifest.json").read_text())

print(f"{len(features):,} events x {features.shape[1]} columns")
print(f"{len(labels):,} labels")
print("dataset fingerprint:", manifest["config_fingerprint"])
print("positive rate (oracle):", manifest["rates"]["positive_rate"])
print("criminal label coverage:", manifest["rates"]["criminal_label_coverage"])

In [ ]:
# The protocol's split, reimplemented here rather than imported: this notebook
# must run standalone on Kaggle, where the kg_* packages are not installed.
# It is checked against the local implementation by
# tests/unit/test_kaggle_parity.py, so the two cannot drift silently.
DAY = 86_400_000
BAND = DAY                     # purge/embargo, protocol section 2
FRACTIONS = (0.60, 0.20, 0.20)

ts = features["acceptance_ts"].to_numpy()
start, end = int(ts.min()), int(ts.max()) + 1
span = end - start
t1 = start + int(span * FRACTIONS[0])
t2 = start + int(span * (FRACTIONS[0] + FRACTIONS[1]))

fold = np.full(len(ts), "test", dtype=object)
fold[ts < t1] = "train"
fold[(ts >= t1) & (ts < t2)] = "validation"
fold[(ts >= t1 - BAND) & (ts < t1 + BAND)] = "dropped"
fold[(ts >= t2 - BAND) & (ts < t2 + BAND)] = "dropped"

# LABEL AVAILABILITY -- the rule that actually stops time travel.
# A label counts for a fold only if it was DECIDED before that fold's boundary.
# Filtering on the transaction's own timestamp is the common mistake and it
# hands the model answers nobody had yet.
POSITIVE = {"CONFIRMED_FRAUD", "CONFIRMED_LAUNDERING"}

def targets(cutoff):
    available = labels if cutoff is None else labels[labels["decided_ts"] < cutoff]
    positive = set(available.loc[available["label"].isin(POSITIVE), "subject_id"])
    return features["tx_id"].isin(positive).to_numpy().astype(np.float32)

y = {
    "train": targets(t1),
    "validation": targets(t2),
    "test": targets(None),   # scored after the fact
}
for name in ("train", "validation", "test"):
    mask = fold == name
    print(f"{name:11} {mask.sum():>8,} rows  {int(y[name][mask].sum()):>5} positive")
print(f"dropped     {(fold == 'dropped').sum():>8,} (purge + embargo)")

In [ ]:
FEATURE_COLUMNS = [
    c for c in features.columns
    if c not in {"tx_id", "acceptance_ts", "debtor_account_id", "creditor_account_id",
                 "amount_cents", "event_date", "source_dataset"}
]
print(f"{len(FEATURE_COLUMNS)} model inputs")

order = np.lexsort((features["tx_id"].to_numpy(), ts))
accounts = sorted(set(features["debtor_account_id"]) | set(features["creditor_account_id"]))
index = {a: i for i, a in enumerate(accounts)}

X = features[FEATURE_COLUMNS].to_numpy(dtype=np.float32)[order]
# Standardised: the time encoder and attention are scale-sensitive, and the raw
# features span cents to milliseconds.
X = (X - X.mean(0)) / (X.std(0) + 1e-6)

src = np.array([index[a] for a in features["debtor_account_id"].to_numpy()[order]], dtype=np.int64)
dst = np.array([index[a] for a in features["creditor_account_id"].to_numpy()[order]], dtype=np.int64)
# Integer SECONDS: PyG's TGNMemory keeps last_update as a Long buffer, and
# millisecond magnitudes push the time encoder into a bad numeric regime.
t_sec = (ts[order] // 1000).astype(np.int64)
fold_ordered = fold[order]
y_ordered = {k: v[order] for k, v in y.items()}
tx_ordered = features["tx_id"].to_numpy()[order]

print(f"graph: {len(accounts):,} nodes, {len(src):,} edges, edge_dim={X.shape[1]}")

In [ ]:
import torch
from torch.nn import Linear, ReLU, Dropout, Sequential
from torch_geometric.nn import TransformerConv
from torch_geometric.nn.models.tgn import (
    TGNMemory, IdentityMessage, LastAggregator, LastNeighborLoader, TimeEncoder,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEMORY_DIM = EMBEDDING_DIM = TIME_DIM = 100
NEIGHBOURS, HEADS, DROPOUT = 10, 2, 0.1
torch.manual_seed(20251009)

class GraphAttentionEmbedding(torch.nn.Module):
    """Temporal attention over a node's most recent neighbours.

    The time delta on each edge is encoded and concatenated to the edge
    features, so attention can tell "paid me a minute ago" from "a month ago".
    Without it this is a static GNN over a recency-truncated neighbourhood.
    """
    def __init__(self, in_channels, out_channels, msg_dim, time_enc):
        super().__init__()
        self.time_enc = time_enc
        self.conv = TransformerConv(in_channels, out_channels // HEADS, heads=HEADS,
                                    dropout=DROPOUT,
                                    edge_dim=msg_dim + time_enc.out_channels)
    def forward(self, x, last_update, edge_index, t, msg):
        rel_t = self.time_enc((last_update[edge_index[0]] - t).to(x.dtype))
        return self.conv(x, edge_index, torch.cat([rel_t, msg], dim=-1))

class EdgeDecoder(torch.nn.Module):
    """Binary classifier over both endpoint embeddings plus the edge features.

    The transaction's own features are included deliberately: the graph supplies
    context, but the payment still carries the amount, channel and VoP outcome,
    and dropping them to look more graph-native would throw away real signal.
    """
    def __init__(self, emb, edge_dim):
        super().__init__()
        self.net = Sequential(Linear(emb * 2 + edge_dim, emb), ReLU(), Dropout(DROPOUT),
                              Linear(emb, emb // 2), ReLU(), Linear(emb // 2, 1))
    def forward(self, z_src, z_dst, edge_feat):
        return self.net(torch.cat([z_src, z_dst, edge_feat], dim=-1)).squeeze(-1)

def build(use_memory=True):
    edge_dim = X.shape[1]
    memory = TGNMemory(
        len(accounts), edge_dim, MEMORY_DIM, TIME_DIM,
        message_module=IdentityMessage(edge_dim, MEMORY_DIM, TIME_DIM),
        aggregator_module=LastAggregator(),
    ).to(DEVICE) if use_memory else None
    time_enc = memory.time_enc if memory is not None else TimeEncoder(TIME_DIM).to(DEVICE)
    emb = GraphAttentionEmbedding(
        MEMORY_DIM if use_memory else edge_dim, EMBEDDING_DIM, edge_dim, time_enc
    ).to(DEVICE)
    dec = EdgeDecoder(EMBEDDING_DIM, edge_dim).to(DEVICE)
    loader = LastNeighborLoader(len(accounts), size=NEIGHBOURS, device=DEVICE)
    return memory, emb, dec, loader

print("device:", DEVICE)

In [ ]:
SRC = torch.as_tensor(src, device=DEVICE)
DST = torch.as_tensor(dst, device=DEVICE)
TS  = torch.as_tensor(t_sec, device=DEVICE)          # Long, matches last_update
FEAT = torch.as_tensor(X, dtype=torch.float, device=DEVICE)

BATCH, EPOCHS, LR, POS_WEIGHT = 200, 5, 1e-4, 50.0

def batches(rows, size=BATCH):
    for i in range(0, len(rows), size):
        yield rows[i:i + size]

def run_training(use_memory=True, epochs=EPOCHS, tag="tgn"):
    memory, emb, dec, loader = build(use_memory)
    modules = [m for m in (memory, emb, dec) if m is not None]
    params = [p for m in modules for p in m.parameters()]
    opt = torch.optim.Adam(params, lr=LR)
    crit = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEVICE))

    train_rows = np.flatnonzero(fold_ordered == "train")
    y_train = torch.as_tensor(y_ordered["train"], device=DEVICE)
    history = []

    for epoch in range(epochs):
        for m in modules: m.train()
        # Reset every epoch: the model must relearn the stream from its start,
        # exactly as it will at deployment.
        if memory is not None: memory.reset_state()
        loader.reset_state()
        total, seen = 0.0, 0

        for rows in batches(train_rows):
            opt.zero_grad()
            bs, bd = SRC[rows], DST[rows]
            nodes = torch.cat([bs, bd]).unique()
            n_id, edge_index, e_id = loader(nodes)
            assoc = torch.empty(len(accounts), dtype=torch.long, device=DEVICE)
            assoc[n_id] = torch.arange(n_id.size(0), device=DEVICE)

            if memory is not None:
                z, last_update = memory(n_id)
            else:
                z = torch.zeros(n_id.size(0), X.shape[1], device=DEVICE)
                last_update = torch.zeros(n_id.size(0), dtype=torch.long, device=DEVICE)

            emb_out = emb(z, last_update, edge_index,
                          TS[rows][0].expand(edge_index.size(1)),
                          FEAT[e_id] if e_id.numel() else FEAT[:0])
            logits = dec(emb_out[assoc[bs]], emb_out[assoc[bd]], FEAT[rows])
            loss = crit(logits, y_train[rows])
            loss.backward(); opt.step()

            # State updated AFTER the step: this batch never contributed to the
            # embedding that predicted it.
            if memory is not None:
                memory.update_state(bs, bd, TS[rows], FEAT[rows]); memory.detach()
            loader.insert(bs, bd)
            total += float(loss) * len(rows); seen += len(rows)

        history.append({"epoch": epoch, "train_loss": total / max(1, seen)})
        print(f"  {tag} epoch {epoch}  loss {total / max(1, seen):.5f}")
    return (memory, emb, dec, loader), history

In [ ]:
@torch.no_grad()
def score(bundle, rows, replay_from=None):
    """Score rows, replaying earlier events first to build the memory they need.

    Resetting and replaying is not optional. Scoring without a reset means the
    test fold is scored against a memory that already absorbed it -- a
    spectacular, meaningless number with no trace in the metrics. Scoring
    without the replay means a cold memory, which understates the model.
    """
    memory, emb, dec, loader = bundle
    for m in (memory, emb, dec):
        if m is not None: m.eval()
    if memory is not None: memory.reset_state()
    loader.reset_state()

    if replay_from is not None:
        for warm in batches(replay_from):
            if memory is not None:
                memory.update_state(SRC[warm], DST[warm], TS[warm], FEAT[warm])
            loader.insert(SRC[warm], DST[warm])

    out = []
    for rows_b in batches(rows):
        bs, bd = SRC[rows_b], DST[rows_b]
        nodes = torch.cat([bs, bd]).unique()
        n_id, edge_index, e_id = loader(nodes)
        assoc = torch.empty(len(accounts), dtype=torch.long, device=DEVICE)
        assoc[n_id] = torch.arange(n_id.size(0), device=DEVICE)
        if memory is not None:
            z, last_update = memory(n_id)
        else:
            z = torch.zeros(n_id.size(0), X.shape[1], device=DEVICE)
            last_update = torch.zeros(n_id.size(0), dtype=torch.long, device=DEVICE)
        emb_out = emb(z, last_update, edge_index,
                      TS[rows_b][0].expand(edge_index.size(1)),
                      FEAT[e_id] if e_id.numel() else FEAT[:0])
        out.append(dec(emb_out[assoc[bs]], emb_out[assoc[bd]], FEAT[rows_b]).cpu().numpy())
        if memory is not None:
            memory.update_state(bs, bd, TS[rows_b], FEAT[rows_b])
        loader.insert(bs, bd)
    return 1 / (1 + np.exp(-np.clip(np.concatenate(out), -50, 50)))

def average_precision(y_true, scores):
    """PR-AUC with explicit tie handling -- the protocol's headline metric."""
    y_true = np.asarray(y_true, dtype=np.int8); scores = np.asarray(scores, dtype=float)
    pos = int(y_true.sum())
    if pos == 0: return 0.0
    o = np.argsort(-scores, kind="stable")
    st, ss = y_true[o], scores[o]
    tp = np.cumsum(st); ranks = np.arange(1, len(st) + 1)
    precision, recall = tp / ranks, tp / pos
    last = np.ones(len(st), bool); last[:-1] = ss[:-1] != ss[1:]
    return float(np.sum(precision[last] * np.diff(np.concatenate([[0.0], recall[last]]))))

def precision_at_k(y_true, scores, k):
    k = min(k, len(y_true))
    return float(np.asarray(y_true)[np.argsort(-scores, kind="stable")[:k]].mean())

## Restore the trained detector

The explainer explains *this* model. Training a fresh one here would explain a different
model than the one that will be served.

In [ ]:
bundle, _ = run_training(use_memory=True, epochs=2, tag="reload")
state = torch.load("/kaggle/input/kontograph-01-train-tgn/tgn.pt", map_location=DEVICE) \
        if Path("/kaggle/input/kontograph-01-train-tgn/tgn.pt").exists() else None
if state is not None:
    memory, emb, dec, loader = bundle
    emb.load_state_dict(state["embedding"]); dec.load_state_dict(state["decoder"])
    if memory is not None and "memory" in state: memory.load_state_dict(state["memory"])
    print("restored the checkpoint from 01")
else:
    print("!! no checkpoint found; using the short retrain above")

train_rows = np.flatnonzero(fold_ordered == "train")
val_rows   = np.flatnonzero(fold_ordered == "validation")
test_rows  = np.flatnonzero(fold_ordered == "test")
base_scores = score(bundle, test_rows, replay_from=np.concatenate([train_rows, val_rows]))

## The perturbation teacher — deliberately slow

In [ ]:
import torch, time, numpy as np

CANDIDATES, STEPS, LR_MASK = 40, 100, 0.05
SIZE_W, ENTROPY_W = 0.05, 0.1

def neighbourhood(row, limit=CANDIDATES):
    """Events strictly EARLIER than the target that touch either endpoint.

    Strictness matters: an explanation citing a later payment would describe
    information the model did not have -- the same leak the feature layer
    forbids, wearing a different hat.
    """
    s, d, t = src[row], dst[row], t_sec[row]
    earlier = np.flatnonzero(t_sec < t)
    if earlier.size == 0: return earlier
    touch = ((src[earlier] == s) | (dst[earlier] == s) |
             (src[earlier] == d) | (dst[earlier] == d))
    cand = earlier[touch]
    if cand.size > limit:
        cand = cand[np.argsort(-t_sec[cand], kind="stable")][:limit]
    return np.sort(cand)

def masked_score(bundle, row, cand, mask):
    """Score the target with candidate edges weighted by `mask`.

    Differentiable in mask, which is what lets the teacher optimise it.
    """
    memory, emb, dec, loader = bundle
    weighted = FEAT[cand] * mask.unsqueeze(-1).to(DEVICE)
    nodes = torch.stack([SRC[row], DST[row]])
    n_id, edge_index, e_id = loader(nodes)
    assoc = torch.empty(len(accounts), dtype=torch.long, device=DEVICE)
    assoc[n_id] = torch.arange(n_id.size(0), device=DEVICE)
    if memory is not None:
        z, last_update = memory(n_id)
    else:
        z = torch.zeros(n_id.size(0), X.shape[1], device=DEVICE)
        last_update = torch.zeros(n_id.size(0), dtype=torch.long, device=DEVICE)
    msg = FEAT[e_id] if e_id.numel() else FEAT[:0]
    if msg.size(0) and weighted.size(0):
        scale = weighted.mean(0, keepdim=True)
        msg = msg * torch.sigmoid(scale).clamp(0.1, 1.0)
    emb_out = emb(z, last_update, edge_index,
                  TS[row].expand(edge_index.size(1)), msg)
    return torch.sigmoid(dec(emb_out[assoc[SRC[row]]].unsqueeze(0),
                             emb_out[assoc[DST[row]]].unsqueeze(0),
                             FEAT[row].unsqueeze(0))).squeeze()

def explain_slow(bundle, row, base):
    """The perturbation teacher: ~100 gradient steps for ONE explanation.

    This cost is the entire premise of the spike (ADR-0007). It is not optimised
    -- optimising it would weaken the fidelity ceiling everything is measured
    against.
    """
    cand = neighbourhood(row)
    if cand.size == 0:
        return cand, np.empty(0), 0.0
    started = time.time()
    logits = torch.nn.Parameter(torch.randn(cand.size, device=DEVICE) * 0.1)
    opt = torch.optim.Adam([logits], lr=LR_MASK)
    target = torch.tensor(float(base), device=DEVICE)
    for _ in range(STEPS):
        opt.zero_grad()
        m = torch.sigmoid(logits)
        fidelity = (masked_score(bundle, row, cand, m) - target) ** 2
        size = SIZE_W * m.mean()
        entropy = -ENTROPY_W * (m * torch.log(m + 1e-9) +
                                (1 - m) * torch.log(1 - m + 1e-9)).mean()
        (fidelity + size + entropy).backward(); opt.step()
    return cand, torch.sigmoid(logits).detach().cpu().numpy(), time.time() - started

## Explain the top alerts

The teacher runs on the highest-scoring alerts, which is where an investigator's
attention actually goes.

In [ ]:
N_EXPLAIN = 60
top = test_rows[np.argsort(-base_scores)[:N_EXPLAIN]]
explained_rows, teacher_candidates, teacher_masks, teacher_times = [], [], [], []

for i, row in enumerate(top):
    cand, mask, seconds = explain_slow(bundle, int(row), base_scores[np.argsort(-base_scores)[i]])
    explained_rows.append(int(row)); teacher_candidates.append(cand)
    teacher_masks.append(mask); teacher_times.append(seconds)
    if i % 20 == 0: print(f"  explained {i}/{N_EXPLAIN} ({seconds:.2f}s each)")

print(f"\nteacher median: {np.median(teacher_times):.2f}s per explanation")
print(f"at that rate a 200ms budget allows {0.2/np.median(teacher_times):.4f} explanations.")
print("That is the problem this spike exists to solve.")

## The amortised student

In [ ]:
from torch.nn import Sequential, Linear, ReLU, Dropout

# The four structural flags student_inputs() builds, named so the head's input
# width is derived from them rather than hand-counted. It was hand-counted, and
# it was wrong: the literal `+ 4` covered the structure block but not `dt`, so
# the head was built 233 wide and fed 234, failing ten minutes into the run with
# "mat1 and mat2 shapes cannot be multiplied (2x234 and 233x64)".
N_STRUCTURE_FEATURES = 4
N_TIME_FEATURES = 1  # log1p(dt)


class ExplainerHead(torch.nn.Module):
    """Edge importances in ONE forward pass, from what the detector already computed."""
    def __init__(self, emb_dim, edge_dim, hidden=64):
        super().__init__()
        # ctx is both endpoints' embeddings concatenated, hence emb_dim * 2.
        self.in_dim = emb_dim * 2 + edge_dim + N_TIME_FEATURES + N_STRUCTURE_FEATURES
        self.net = Sequential(Linear(self.in_dim, hidden), ReLU(), Dropout(0.1),
                              Linear(hidden, hidden // 2), ReLU(), Linear(hidden // 2, 1))

    def forward(self, ctx, cand_feat, dt, structure):
        n = cand_feat.size(0)
        x = torch.cat([ctx.expand(n, -1), cand_feat, dt.unsqueeze(-1), structure], -1)
        # Assert rather than let the matmul report it. A shape mismatch here is a
        # bug in how the inputs are assembled, and the matmul error names the
        # matrices instead of the cause.
        if x.size(-1) != self.in_dim:
            raise RuntimeError(
                f"ExplainerHead built for {self.in_dim} inputs but received {x.size(-1)}: "
                f"ctx={ctx.size(-1)}, cand_feat={cand_feat.size(-1)}, dt=1, "
                f"structure={structure.size(-1)}"
            )
        return self.net(x).squeeze(-1)

def student_inputs(bundle, row, cand):
    memory, emb, dec, loader = bundle
    with torch.no_grad():
        nodes = torch.stack([SRC[row], DST[row]])
        n_id, edge_index, e_id = loader(nodes)
        assoc = torch.empty(len(accounts), dtype=torch.long, device=DEVICE)
        assoc[n_id] = torch.arange(n_id.size(0), device=DEVICE)
        z = memory(n_id)[0] if memory is not None else torch.zeros(n_id.size(0), X.shape[1], device=DEVICE)
        lu = memory(n_id)[1] if memory is not None else torch.zeros(n_id.size(0), dtype=torch.long, device=DEVICE)
        out = emb(z, lu, edge_index, TS[row].expand(edge_index.size(1)),
                  FEAT[e_id] if e_id.numel() else FEAT[:0])
        ctx = torch.cat([out[assoc[SRC[row]]], out[assoc[DST[row]]]]).unsqueeze(0)
    # log1p seconds: the interesting range spans seconds to weeks, and a linear
    # encoding would make everything inside an hour indistinguishable -- which
    # is exactly where laundering velocity lives.
    dt = torch.as_tensor(np.log1p(np.maximum(t_sec[row] - t_sec[cand], 0)),
                         dtype=torch.float, device=DEVICE)
    s, d = src[row], dst[row]
    structure = torch.as_tensor(np.stack([
        (src[cand] == s) | (dst[cand] == s),
        (src[cand] == d) | (dst[cand] == d),
        dst[cand] == s,                       # forms a chain: money in, then out
        np.ones(cand.size),
    ], -1).astype(np.float32), device=DEVICE)
    return ctx, FEAT[cand], dt, structure

## Distil

In [ ]:
head = ExplainerHead(EMBEDDING_DIM, X.shape[1]).to(DEVICE)
opt = torch.optim.Adam(head.parameters(), lr=1e-3)

examples = [(r, c, torch.as_tensor(m, dtype=torch.float, device=DEVICE))
            for r, c, m in zip(explained_rows, teacher_candidates, teacher_masks) if c.size]
for epoch in range(30):
    total = 0.0
    for row, cand, teach in examples:
        opt.zero_grad()
        pred = head(*student_inputs(bundle, row, cand))
        loss = torch.nn.functional.binary_cross_entropy_with_logits(pred, teach)
        loss.backward(); opt.step(); total += float(loss)
    if epoch % 10 == 0:
        print(f"  epoch {epoch}: distillation loss {total/max(1,len(examples)):.5f}")

## The fidelity gap — published either way

In [ ]:
def top_k_jaccard(a, b, k=5):
    A, B = set(np.argsort(-a)[:k]), set(np.argsort(-b)[:k])
    return len(A & B) / len(A | B) if (A or B) else 1.0

def spearman(a, b):
    if a.size < 2: return 0.0
    ra, rb = np.argsort(np.argsort(a)).astype(float), np.argsort(np.argsort(b)).astype(float)
    ra -= ra.mean(); rb -= rb.mean()
    den = np.sqrt((ra**2).sum() * (rb**2).sum())
    return float((ra * rb).sum() / den) if den else 0.0

# The fidelity gap. ADR-0007 commits to publishing it whether or not it flatters.
#
# NOTE the absent `teacher_times` here. It is measured in the cell that runs the
# teacher, and re-initialising it in this line wiped those timings -- nothing
# appends to it below, so the median was taken over an empty list and the
# reported speedup came out NaN. The teacher's real median (0.62 s) survived
# only because the loop printed it.
jaccards, correlations, student_times = [], [], []
n_candidates = []
for row, cand, teach in zip(explained_rows, teacher_candidates, teacher_masks):
    if cand.size == 0: continue
    started = time.time()
    with torch.no_grad():
        pred = torch.sigmoid(head(*student_inputs(bundle, row, cand))).cpu().numpy()
    student_times.append((time.time() - started) * 1000)
    jaccards.append(top_k_jaccard(pred, teach))
    correlations.append(spearman(pred, teach))
    n_candidates.append(int(cand.size))

print(f"teacher: {np.median(teacher_times)*1000:8.1f} ms per explanation")
print(f"student: {np.median(student_times):8.3f} ms per explanation")
print(f"speedup: {np.median(teacher_times)*1000/max(np.median(student_times),1e-6):,.0f}x")
print()
print(f"top-5 Jaccard vs teacher: {np.mean(jaccards):.4f}")
print(f"Spearman vs teacher:      {np.mean(correlations):.4f}")
# Read the Jaccard against this. With a median candidate set near k, "the top 5
# of 6" overlaps heavily no matter what the model learned, and the set metric
# flatters while the rank correlation tells the truth.
print(f"median candidates per instance: {np.median(n_candidates):.1f} (k=5)")
print()
print("This gap is the headline of the write-up. A fast explainer that does not")
print("track the teacher is fast and unfaithful, which is worse than none at all.")

## Write the artifact bundle

In [ ]:
torch.save(head.state_dict(), "/kaggle/working/explainer_head.pt")
Path("/kaggle/working/metrics.json").write_text(json.dumps({
    "explained_instances": len(examples),
    "teacher_median_seconds": float(np.median(teacher_times)),
    "median_candidates": float(np.median(n_candidates)),
    "student_median_ms": float(np.median(student_times)),
    "speedup": float(np.median(teacher_times) * 1000 / max(np.median(student_times), 1e-6)),
    "top5_jaccard_vs_teacher": float(np.mean(jaccards)),
    "spearman_vs_teacher": float(np.mean(correlations)),
}, indent=2))
print("wrote /kaggle/working/metrics.json and explainer_head.pt")